## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [15]:
def flow_rate(weight, time, period=60, units_per_kg=1000):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [2]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [ ]:
Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

In [17]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [20]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [21]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

1e+04 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

In [25]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [28]:
from time import time

def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    def wrapper(n):
        start = time()
        res = func(n)
        end = time()
        
        print("Function {} took: {:.5f} seconds".format(
            func.__name__, end - start
        ))

        return res
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.37729 seconds
Function squares_comprehension took: 0.39689 seconds
Function squares_map took: 0.00000 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [30]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
    """ Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h, 
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy """

    if epsilon is None:
        epsilon = 1000 * sys.float_info.epsilon
    def decorator(func):
        def wrapper(x):
            first = func(x + epsilon)
            second = func(x)
            res = (first - second) / epsilon
            return res
        return wrapper
    return decorator


@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
637.5213417035411 637.5113417049123 	 True
304815.54195284843 304815.5331017981 	 True
348.4108609769464 348.4008609771165 	 True
91037.37520053981 91037.36994719705 	 True
1982.5867176870815 1982.5767176953323 	 True
2947957.849502563 2947957.831160698 	 True
1826.9493043539114 1826.9393043620662 	 True
2503280.4250717163 2503280.416367213 	 True
1416.504550020909 1416.4945500224546 	 False
1504842.6270484922 1504842.6076824872 	 True
1501.6629459918477 1501.6529459983456 	 True
1691221.1954593656 1691221.1776691326 	 True
1392.5805001577828 1392.5705001597873 	 True
1454439.467191696 1454439.4484364602 	 True
361.3333662480727 361.3233662480959 	 True
97915.93672707675 97915.93124764174 	 True
13.639462385185652 13.629462385185587 	 True
139.32188811622837 139.32168368189159 	 True
843.5552583832759 843.5452583828229 	 True
533676.4633655548 533676.4522051075 	 True
243.24435240596358 243.23435240578027 	 True
44372.21633270383 44372.212642694474 	 True
1944.07820

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [34]:
from functools import wraps
import inspect

def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane 
    w argumentach dekoratora"""

    def decorator(func):
        def wrapper(*args, **kwargs):
            all = args + tuple(kwargs.values())
            for arg, expected in zip(all, types):
                if not isinstance(arg, expected):
                    raise TypeError(f"Oczekiwano {expected}, dostano {type(arg)}")
            return func(*args, **kwargs)
        return wrapper
    return decorator



@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [37]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)

            if len(types) == 1:
                results = (result,)
            else:
                results = result

            if len(results) != len(types):
                raise TypeError(f"Oczekiwano {len(types)} wartości, zwrócono {len(results)}")
            
            for res, expected in zip(results, types):
                if not isinstance(res, expected):
                    raise TypeError(f"Zły typ zwracanej wartości! Oczekiwano {expected}, dostano {type(res)}")
                
            return result
        return wrapper
    return decorator

@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [38]:
from random import random
from functools import wraps

def cached(func):
    cache = {}
    called = 0
    evaluated = 0

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal called, evaluated

        called += 1
        
        key = (args, tuple(sorted(kwargs.items())))
        if key in cache:
            return cache[key]
        
        evaluated += 1
        res = func(*args, **kwargs)
        cache[key] = res

        return res
    
    def cache_reset():
        cache.clear()
    
    def cache_status():
        return f"Function {func.__name__} called {called} times, evaluated {evaluated} times"

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status

    return wrapper




@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [39]:
from functools import wraps

def zlozenie(n):
    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):
            res = func(*args, **kwargs)
            
            for _ in range(n - 1):
                res = func(res)
            
            return res
        return wrapper
    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [44]:
def my_function(x, y):
    cases = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0,
    }

    return cases[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
